<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

This installs the libraries needed to run the notebook.

In [ ]:
!pip install pandas

## Imports and setup

We import the yFinance, which gives us one function call for historical daily closes. pandas arrives with the returned DataFrame, so we use its methods without importing it separately.

In [ ]:
import yfinance as yf

One import keeps the data source swappable later. If you move to a broker feed, everything after this point still works as long as you end up with a DataFrame of daily closes.

## Download prices and compute daily returns

We pull daily closes for two holdings and the benchmark back to the start of 2020.

In [ ]:
prices = yf.download(
    ["AAPL", "WMT", "SPY"],
    start="2020-01-01",
)

Requesting SPY in the same call as the holdings means the benchmark shares the exact same dates, so we never have to merge two frames and risk misaligned rows. A one day offset between portfolio and benchmark will quietly change the capture number, and it is hard to spot once the result is a single float.

We convert prices into daily percentage changes, which is what the capture calculation compares.

In [ ]:
returns = prices.Close.pct_change().dropna()

Capture compares percentage moves, not dollar levels, so prices are useless to us until this step. The dropna matters because pct_change leaves a NaN in the first row where there is no prior close, and that NaN would wipe out the compounded product we calculate later.

## Build a portfolio return series

We add a portfolio column by combining the two holdings, then keep only the benchmark and the portfolio.

In [ ]:
data = returns.assign(
    port=returns[["AAPL", "WMT"]].sum(axis=1)
)[["SPY", "port"]]

The sum treats AAPL and WMT as two full-size positions rather than a 50/50 split, so this portfolio runs twice the gross exposure of SPY. Expect a capture ratio well above 100% as a result, which is exactly the leverage signal the post describes. Swap sum for mean if you want an equal weighted portfolio instead.

## Calculate the up-market capture ratio

This turns any series of daily returns into a compounded annual rate using 252 trading days per year.

In [ ]:
def annual_return(returns):
    num_years = len(returns) / 252
    return (returns + 1).prod() ** (1 / num_years) - 1

We annualize both sides so the portfolio and the benchmark are measured on the same scale before we divide. Note that when we hand this function a filtered subset, num_years reflects only the days in that subset, which keeps the two numbers comparable.

This filters both series down to the days the benchmark rose, then divides the annualized portfolio return by the annualized benchmark return.

In [ ]:
def upside_capture(port_returns, bench_returns):
    mask = bench_returns > 0
    port_returns = port_returns[mask]
    bench_returns = bench_returns[mask]

    return annual_return(port_returns) / annual_return(bench_returns)

The mask comes from the benchmark and gets applied to both series, which is the whole point. We are asking what our portfolio did on the specific days SPY went up, not what it did on its own good days. Flip the comparison to bench_returns < 0 and you get down capture, where a result under 100% means we lost less than the index when it fell.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.